# What do sequence models learn about DM and HSR chromatin?

**The data.** The same ATAC-seq as Step 3: COLO320DM (*MYC* on ecDNA) and COLO320HSR (*MYC* in a
chromosome), two replicates each.

**The models.** Two deep-learning tools that read DNA sequence and predict ATAC signal:

- **chromBPNet** predicts how many reads a region gets and where Tn5 cuts inside it;
- **seq2PRINT** predicts **footprints**: the dips in Tn5 cutting where proteins sit on DNA, at 99 widths
  from TF-sized to nucleosome-sized.

Each line has its own models. Training took days of GPU time, so this notebook starts from the trained
models and their outputs. You can still run the models live: edit DNA and see what they predict.

> **In progress.** One model per line and tool, each trained on both replicates pooled, with fold 0 of
> chromBPNet's standard train/test chromosome splits. Sections 1 and 3 need the models' evaluation and TF-MoDISco
> results, which are still being added; until then those two cells print a note and move on. Sections 5-7 are
> still to come.

**The plan.**

1. Did the models train well?
2. ✏️ **Explore a peak**: which bases does each model rely on?
3. Which motifs did the chromBPNet models learn?
4. ✏️ **Edit the DNA**: remove or plant a motif, and watch both models respond.
5. DM vs HSR, and agreement with HOMER (to come).

*To run in Colab: Runtime → Change runtime type → **Python 3** (a CPU is enough), then run the cells in order.*

## 0. Setup (about 2 minutes)

In [ ]:
# Install logomaker (sequence logos); TensorFlow, PyTorch and the rest come with Colab
import importlib.util, subprocess, sys
if importlib.util.find_spec("logomaker") is None:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "logomaker"], check=True)

In [ ]:
# Download the data (one archive on Google Drive, about 230 MB) and unpack it into data/
import os, tarfile, urllib.request
drive_file_id = "TODO_PASTE_FILE_ID"      # from the Drive share link: https://drive.google.com/file/d/<FILE_ID>/view
if not os.path.exists("data/regions.npz"):
    url = f"https://drive.usercontent.google.com/download?id={drive_file_id}&export=download&confirm=t"
    urllib.request.urlretrieve(url, "ai_models_data.tar.gz")
    tarfile.open("ai_models_data.tar.gz").extractall(".")
sorted(os.listdir("data")), sorted(os.listdir("data/models"))

In [ ]:
# Load everything and define helper functions (no need to read this cell closely)
import glob, json, warnings, contextlib
import numpy as np, pandas as pd, matplotlib.pyplot as plt, logomaker
warnings.filterwarnings("ignore")
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"; os.environ["TF_ENABLE_ONEDNN_OPTS"] = "0"

@contextlib.contextmanager
def quiet():                       # TensorFlow prints start-up messages straight to stderr
    fd = os.dup(2); null = os.open(os.devnull, os.O_WRONLY); os.dup2(null, 2)
    try:
        yield
    finally:
        os.dup2(fd, 2); os.close(null); os.close(fd)

with quiet():
    import tensorflow as tf, torch
plt.rcParams.update({"font.size": 13, "axes.spines.top": False, "axes.spines.right": False})
COLORS = {"DM": "firebrick", "HSR": "steelblue"}

# featured peaks: 250 more accessible in DM, 250 in HSR, 250 unchanged (Step 3's calls, copy number in the model)
d = np.load("data/regions.npz", allow_pickle=True)
TRACKS = {name: d["tracks"][:, i].astype(np.float32) for i, name in enumerate(d["track_names"])}
peaks = pd.DataFrame({k: d[k] for k in ["peak", "chrom", "center", "gene", "call", "log2FC", "padj"]})
peaks["rank"] = peaks.groupby("call").cumcount() + 1           # 1 = most significant in its group
SEQS = d["seq"]                                                 # 2,114 bp around each peak center
POS = np.arange(-500, 500)                                      # tracks cover the central 1,000 bp

# models: every file found per tool and line is used, and their predictions averaged
with quiet():
    CBP = {line: [tf.keras.models.load_model(f, compile=False) for f in sorted(glob.glob(f"data/models/chrombpnet_{line}_*.h5"))]
           for line in ["DM", "HSR"]}
S2P = {line: [torch.jit.load(f, map_location="cpu").eval() for f in sorted(glob.glob(f"data/models/seq2print_{line}_*.pt"))]
       for line in ["DM", "HSR"]}
torch.set_num_threads(2)

def one_hot(seq):
    x = np.zeros((len(seq), 4), dtype=np.float32)
    for i, b in enumerate(seq):
        j = "ACGT".find(b)
        if j >= 0:
            x[i, j] = 1
    return x

def predict(seqs):
    """Predictions of both tools for 2,114 bp sequences, averaged over each line's models."""
    X = np.stack([one_hot(s) for s in seqs])
    Xs = torch.tensor(X[:, 137:137 + 1840].transpose(0, 2, 1).copy())    # seq2PRINT reads the central 1,840 bp
    out = {}
    for line in ["DM", "HSR"]:
        preds = [m.predict(X, verbose=0) for m in CBP[line]]
        out[f"chrombpnet_logcounts_{line}"] = np.mean([p[1][:, 0] for p in preds], axis=0)
        with torch.no_grad():
            s2p = [m(Xs) for m in S2P[line]]
        out[f"seq2print_footprint_{line}"] = np.mean([o[0].numpy() for o in s2p], axis=0)   # (n, 99 widths, 800 bp)
        out[f"seq2print_count_{line}"] = np.mean([o[1].numpy() for o in s2p], axis=0)
    return out

def pct(pred, tool, line, i_after=1, i_before=0):
    # percent change in predicted reads (both tools predict on a log scale)
    key = f"{tool}_logcounts_{line}" if tool == "chrombpnet" else f"seq2print_count_{line}"
    return 100 * (np.exp(np.mean(pred[key][i_after] - pred[key][i_before])) - 1)

def find_peak(call=None, rank=1, gene=None):
    hit = peaks[peaks.gene == gene] if gene else peaks[(peaks.call == call) & (peaks["rank"] == rank)]
    if len(hit) == 0:
        raise ValueError("no such peak in the featured set; see the table in section 2")
    return hit.index[0]

def logo(ax, contrib, seq, title):
    df = pd.DataFrame(0.0, index=range(len(seq)), columns=list("ACGT"))
    for i, b in enumerate(seq):
        if b in "ACGT":
            df.loc[i, b] = contrib[i]
    logomaker.Logo(df, ax=ax, color_scheme="classic")
    ax.set_title(title, fontsize=12, loc="left"); ax.set_xticks([])

print("peaks:", peaks.call.value_counts().to_dict())
print("models per line:", {l: (len(CBP[l]), len(S2P[l])) for l in CBP}, "(chromBPNet, seq2PRINT)")

## 1. Did the models train well?

Each chromBPNet model was tested on chromosomes it never saw during training (chr1, chr3 and chr6 for
fold 0).

- **counts_pearson_r**: predicted vs observed reads per peak. 0.6–0.8 is typical for bulk ATAC.
- **profile_median_norm_jsd**: how well the predicted *shape* of the cut pattern matches (0 = no better
  than flat, 1 = perfect). It depends strongly on sequencing depth.

TODO: seq2PRINT validation metrics.

In [ ]:
if os.path.exists("data/qc_chrombpnet.tsv"):
    display(pd.read_csv("data/qc_chrombpnet.tsv", sep="\t"))
else:
    print("TODO: the models' evaluation isn't in this data bundle yet")

## 2. Explore a peak: which bases does each model rely on?

For one peak, from top to bottom:

- **ATAC coverage** in each line (the data);
- **chromBPNet contribution scores**: for each base, how much it pushes the predicted accessibility up
  (above the line) or down (below);
- **seq2PRINT contribution scores**: the same idea for seq2PRINT (on a different scale);
- **seq2PRINT TF binding score**: the probability (0–1) that some TF is bound at each position (computed
  for the central 800 bp).

Then a zoom, where each base is drawn as its letter, scaled by its contribution. A bound TF motif shows up
as a block of tall letters.

**✏️ Try it.** Pick a peak by group and rank (`"up in DM"`, `"up in HSR"` or `"not significant"`; rank 1 is
the most significant), or by `gene`. Move the zoom with `zoom_center` (position relative to the peak
center; `None` = the base with the largest contribution).

In [ ]:
peaks.groupby("call").head(5)[["call", "rank", "gene", "chrom", "center", "log2FC"]]

In [ ]:
call, rank, gene = "up in HSR", 1, None
zoom_center, zoom_width = None, 60

i = find_peak(call, rank, gene)
p = peaks.loc[i]
fig, axes = plt.subplots(4, 1, figsize=(14, 9), sharex=True)
for line in ["DM", "HSR"]:
    axes[0].plot(POS, TRACKS[f"coverage_{line}"][i], color=COLORS[line], label=line)
    axes[1].plot(POS, TRACKS[f"chrombpnet_contrib_{line}"][i], color=COLORS[line], lw=0.8)
    axes[2].plot(POS, TRACKS[f"seq2print_contrib_{line}"][i], color=COLORS[line], lw=0.8)
    axes[3].plot(POS, TRACKS[f"seq2print_tfbs_{line}"][i], color=COLORS[line])
for ax, lab in zip(axes, ["ATAC coverage", "chromBPNet\ncontribution", "seq2PRINT\ncontribution", "seq2PRINT\nTF binding"]):
    ax.set_ylabel(lab)
axes[0].legend(frameon=False); axes[3].set_xlabel("position relative to the peak center (bp)")
axes[0].set_title(f"{p.gene} ({p.chrom}:{p.center:,}), {p.call}, log2FC DM/HSR = {p.log2FC:.2f}", loc="left")
if zoom_center is None:
    both = np.abs(TRACKS["chrombpnet_contrib_DM"][i]) + np.abs(TRACKS["chrombpnet_contrib_HSR"][i])
    zoom_center = int(POS[np.argmax(both)])
for ax in axes:
    ax.axvspan(zoom_center - zoom_width // 2, zoom_center + zoom_width // 2, color="gold", alpha=0.3)
plt.show()

# zoom: letters scaled by contribution
lo = 500 + zoom_center - zoom_width // 2
seq = SEQS[i][1057 - 500 + lo:1057 - 500 + lo + zoom_width]
fig, axes = plt.subplots(4, 1, figsize=(14, 7))
for k, (tool, line) in enumerate([(t, l) for t in ["chrombpnet", "seq2print"] for l in ["DM", "HSR"]]):
    logo(axes[k], TRACKS[f"{tool}_contrib_{line}"][i][lo:lo + zoom_width], seq,
         f"{'chromBPNet' if tool == 'chrombpnet' else 'seq2PRINT'}, {line} model")
plt.tight_layout(); plt.show()
print("zoomed sequence:", seq)

## 3. Motifs the chromBPNet models learned

**TF-MoDISco** gathers the stretches with the highest contributions across all peaks, clusters them into
motifs, and matches each to a database of known TF motifs. **Seqlets** is how many stretches support a
motif. Each list is complete (roughly 40–50 motifs per line), from a few very common motifs to many rare
ones.

This describes each line's motif vocabulary. It is not a DM-vs-HSR comparison: motifs are clustered
separately for each model, one TF can be split into several motifs or merged into a similar one, and
seqlet counts depend on each model's data.

**✏️ Try it.** Change `line`, `top_n`, or search the database matches with `search` (a regular expression,
e.g. `"CTCF"`, `"JUN|FOS"`, `"CDX|HOX"`). Always check the logo, not just the name.

In [ ]:
if not os.path.exists("data/motifs_chrombpnet.json"):
    print("TODO: the TF-MoDISco results aren't in this data bundle yet")
else:
    MOTIFS = json.load(open("data/motifs_chrombpnet.json"))
    line, top_n, search = "HSR", 12, None

    pats = MOTIFS[line]
    if search:
        pats = [m for m in pats if any(pd.Series([name for name, q in m["matches"]]).str.contains(search, case=False))]
    pats = pats[:top_n]
    ncol = 3; nrow = max(1, int(np.ceil(len(pats) / ncol)))
    fig, axes = plt.subplots(nrow, ncol, figsize=(15, 2.2 * nrow), squeeze=False)
    for ax, m in zip(axes.flat, pats):
        logomaker.Logo(pd.DataFrame(m["cwm"], columns=list("ACGT")), ax=ax, color_scheme="classic")
        best = m["matches"][0][0] if m["matches"] else "no match"
        ax.set_title(f"{m['pattern'].replace('_patterns.pattern_', ' #')}: {m['n_seqlets']:,} seqlets\nbest match: {best}", fontsize=10, loc="left")
        ax.set_xticks([]); ax.set_yticks([])
    for ax in axes.flat[len(pats):]:
        ax.axis("off")
    plt.suptitle(f"{line} model: {len(MOTIFS[line])} motifs in total", y=1.0); plt.tight_layout(); plt.show()

## 4. Edit the DNA and watch both models respond

The models run live here, on Colab's CPU. First, **remove a motif**: replace the stretch with the largest
contribution in a peak by random bases, and predict before and after with both tools and both lines'
models.

- **chromBPNet**: predicted reads in the peak (the percentage change is what matters);
- **seq2PRINT**: predicted footprints at every width (y axis, TF-sized at the bottom, nucleosome-sized at
  the top) along the peak, before, after, and the difference.

**✏️ Try it.** Choose the peak, the `line` whose seq2PRINT footprints to draw, or set `edit_start` (position
relative to the peak center) and `edit_length` yourself.

In [ ]:
call, rank, gene = "up in HSR", 1, None
edit_start, edit_length = None, 12          # None = centered on the base with the largest chromBPNet contribution
line = "HSR"

i = find_peak(call, rank, gene)
if edit_start is None:
    both = np.abs(TRACKS["chrombpnet_contrib_DM"][i]) + np.abs(TRACKS["chrombpnet_contrib_HSR"][i])
    edit_start = int(POS[np.argmax(both)]) - edit_length // 2
before = SEQS[i]
a = 1057 + edit_start
rng = np.random.default_rng(0)
after = before[:a] + "".join(rng.choice(list("ACGT"), edit_length)) + before[a + edit_length:]
print("original:", before[a - 10:a], before[a:a + edit_length], before[a + edit_length:a + edit_length + 10])
print("edited:  ", after[a - 10:a], after[a:a + edit_length], after[a + edit_length:a + edit_length + 10])

pred = predict([before, after])
display(pd.DataFrame({f"{m} model": {"chromBPNet: predicted reads, change (%)": pct(pred, "chrombpnet", m),
                                      "seq2PRINT: predicted reads, change (%)": pct(pred, "seq2print", m)}
                      for m in ["DM", "HSR"]}).round(1))

fp_before, fp_after = pred[f"seq2print_footprint_{line}"]
x = np.arange(-400, 400)
fig, axes = plt.subplots(1, 3, figsize=(16, 4), sharey=True)
lo, hi = min(fp_before.min(), fp_after.min()), max(fp_before.max(), fp_after.max())
for ax, img, title in zip(axes, [fp_before, fp_after, fp_after - fp_before], ["before", "after", "after - before"]):
    diff = title == "after - before"
    lim = np.abs(img).max()
    im = ax.imshow(img, aspect="auto", origin="lower", extent=[x[0], x[-1], 0, img.shape[0]],
                   cmap="RdBu_r" if diff else "coolwarm", vmin=-lim if diff else lo, vmax=lim if diff else hi)
    ax.axvline(edit_start, color="white" if title != "after - before" else "black", lw=0.8)
    ax.set_title(f"seq2PRINT footprints, {line} model: {title}", fontsize=11); ax.set_xlabel("bp from peak center")
    plt.colorbar(im, ax=ax, fraction=0.04)
axes[0].set_ylabel("footprint width (small to large)")
plt.tight_layout(); plt.show()

Now the reverse: **plant a motif** in the middle of 16 random DNA sequences (41% GC, like the genome) and
measure how much each line's models say accessibility rises, averaged over the 16. Random DNA has no
motifs of its own, so the effect is the motif's alone. (About 25 seconds per motif.)

**✏️ Try it.** `motif` can be any sequence: CTCF `CCGCGAGGGGGCAG`, CRE `TGACGTCA`, CDX `TTTATTGC`, NF-Y
`CCAATCAG`, AP-1 `TGAGTCA`, or one you invent. Controls matter: try the same letters scrambled, and two
copies (`TGAGTCAGGATGAGTCA`).

In [ ]:
motif = "CCGCGAGGGGGCAG"

rng = np.random.default_rng(0)
backgrounds = ["".join(rng.choice(list("ACGT"), 2114, p=[0.295, 0.205, 0.205, 0.295])) for _ in range(16)]
a = 1057 - len(motif) // 2
planted = [s[:a] + motif.upper() + s[a + len(motif):] for s in backgrounds]
pred = predict(backgrounds + planted)
n = len(backgrounds)
effect = {f"{m} model": {tool_name: 100 * (np.exp(np.mean(pred[key.format(m)][n:] - pred[key.format(m)][:n])) - 1)
                         for tool_name, key in [("chromBPNet: predicted reads, change (%)", "chrombpnet_logcounts_{}"),
                                                ("seq2PRINT: predicted reads, change (%)", "seq2print_count_{}")]}
          for m in ["DM", "HSR"]}
pd.DataFrame(effect).round(1)

## 5. DM vs HSR, and do the tools agree?

Each model describes one line, so a DM-vs-HSR difference means comparing the DM models with the HSR models
on a common scale:

- **Motif insertion effect**: insert a motif into thousands of background sequences and measure how much
  each model's predicted accessibility rises (what you just did for one peak, done systematically with
  chromBPNet `footprints` and seq2PRINT `delta_effects_seq2print`). One number per motif per model.
- **The same sites in both models**: at every genomic site of a motif, compare the DM and HSR models'
  contribution scores.

Then set these beside HOMER's result from Step 3, and beside each other.

TODO: results. Note: DM and HSR were sequenced in separate batches, so any difference here, as in Step 3,
can partly be batch.

## 6. Going deeper (optional): how much do models disagree?

TODO: compare the models trained on each replicate separately (both tools also have one model per replicate).

## 7. Next steps for interpreting these outputs

TODO. Candidates: predict every peak with the DM and HSR models and compare the predicted fold change with
DESeq2's; find which motifs drive the predicted differences.